# Image Mosaicing Pipeline
**Sections covered:**
- **2.1** — Collect correspondences interactively + compute homography
- **2.2** — Inverse warp with bilinear interpolation
- **2.3** — Compose the output mosaic

## Configuration
Set your image paths and number of correspondence points here.

In [ ]:
# ── USER CONFIG ──────────────────────────────────────────────
IMG_SRC_PATH   = "images/left.jpg"    # source image (will be warped)
IMG_DST_PATH   = "images/right.jpg"   # destination / reference image
N_POINTS       = 6                    # number of correspondence pairs (>= 4)
OUTPUT_PATH    = "outputs/mosaic.png" # where to save the final mosaic
# ─────────────────────────────────────────────────────────────

## Imports

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

%matplotlib inline
print("Imports OK")

---
## 2.1 — Get Correspondences & Compute Homography

In [ ]:
img_src = mpimg.imread(IMG_SRC_PATH)
img_dst = mpimg.imread(IMG_DST_PATH)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(img_src); axes[0].set_title("Source");      axes[0].axis("off")
axes[1].imshow(img_dst); axes[1].set_title("Destination"); axes[1].axis("off")
plt.tight_layout()
plt.show()
print(f"Source shape      : {img_src.shape}")
print(f"Destination shape : {img_dst.shape}")

In [ ]:
def get_correspondences(img1, img2, n_points=4):
    """
    Interactively collect n_points correspondence pairs.

    Left panel  → img1 (source)
    Right panel → img2 (destination)

    Returns
    -------
    pts1, pts2 : (n, 2) arrays in (x, y) order
    """
    # Switch to an interactive backend so ginput works inside the notebook.
    matplotlib.use("TkAgg")   # change to 'Qt5Agg' if Tk is unavailable

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(img1); axes[0].set_title("Source – click here first")
    axes[1].imshow(img2); axes[1].set_title("Destination – click here second")
    plt.suptitle(
        f"Click {n_points} points in the SOURCE, "
        f"then {n_points} matching points in the DESTINATION.",
        fontsize=11
    )
    plt.tight_layout()

    all_clicks = plt.ginput(n=2 * n_points, timeout=0, show_clicks=True)
    plt.close(fig)
    matplotlib.use("inline")   # restore inline rendering

    pts1 = np.array(all_clicks[:n_points])
    pts2 = np.array(all_clicks[n_points:])
    return pts1, pts2


pts_src, pts_dst = get_correspondences(img_src, img_dst, n_points=N_POINTS)
print("Source points (x, y):")
print(pts_src)
print("\nDestination points (x, y):")
print(pts_dst)

In [ ]:
# Visualise collected correspondences
colors = plt.cm.tab10(np.linspace(0, 1, N_POINTS))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, img, pts, title in [
    (axes[0], img_src, pts_src, "Source correspondences"),
    (axes[1], img_dst, pts_dst, "Destination correspondences"),
]:
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")
    for i, (x, y) in enumerate(pts):
        ax.plot(x, y, "o", color=colors[i], markersize=8)
        ax.text(x + 5, y - 5, str(i + 1), color=colors[i], fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
def compute_homography(pts1, pts2):
    """
    Compute 3x3 homography H such that  pts2 ~ H @ pts1  (homogeneous coords).

    Parameters
    ----------
    pts1, pts2 : (n, 2) arrays of corresponding (x, y) points, n >= 4

    Returns
    -------
    H : (3, 3) homography matrix, normalised so H[2,2] == 1
    """
    n = len(pts1)
    assert n >= 4, "Need at least 4 point pairs."

    A = []
    for (x, y), (xp, yp) in zip(pts1, pts2):
        A.append([-x, -y, -1,  0,  0,  0, x * xp, y * xp, xp])
        A.append([ 0,  0,  0, -x, -y, -1, x * yp, y * yp, yp])

    A = np.array(A, dtype=float)   # (2n, 9)
    _, _, Vt = np.linalg.svd(A)
    h = Vt[-1]                     # eigenvector of smallest singular value
    H = h.reshape(3, 3)
    H /= H[2, 2]                   # normalise
    return H


H = compute_homography(pts_src, pts_dst)

print("Homography matrix H:")
print(np.round(H, 6))

# Reprojection error
ones  = np.ones((N_POINTS, 1))
ph    = np.hstack([pts_src, ones]).T      # (3, n)
proj  = H @ ph
proj  = (proj[:2] / proj[2]).T            # (n, 2)
error = np.linalg.norm(proj - pts_dst, axis=1)
print(f"\nMean reprojection error : {error.mean():.3f} px")
print(f"Max  reprojection error : {error.max():.3f} px")

---
## 2.2 — Warp Image (Inverse Warp + Bilinear Interpolation)

In [ ]:
def _bilinear_sample(img, x_coords, y_coords):
    """
    Sample a single-channel float image at sub-pixel positions using
    bilinear interpolation.  Out-of-bounds positions return 0.

    Parameters
    ----------
    img      : (H, W) float64 array
    x_coords : (N,) float array — column positions
    y_coords : (N,) float array — row positions

    Returns
    -------
    values : (N,) float64 array
    """
    h, w = img.shape

    x0 = np.floor(x_coords).astype(int)
    y0 = np.floor(y_coords).astype(int)
    x1 = x0 + 1
    y1 = y0 + 1

    dx = x_coords - x0   # fractional part
    dy = y_coords - y0

    x0c = np.clip(x0, 0, w - 1)
    x1c = np.clip(x1, 0, w - 1)
    y0c = np.clip(y0, 0, h - 1)
    y1c = np.clip(y1, 0, h - 1)

    valid = (x_coords >= 0) & (x_coords < w) & (y_coords >= 0) & (y_coords < h)

    I00 = img[y0c, x0c]
    I10 = img[y0c, x1c]
    I01 = img[y1c, x0c]
    I11 = img[y1c, x1c]

    values = ((1 - dy) * (1 - dx) * I00 +
              (1 - dy) *      dx  * I10 +
                   dy  * (1 - dx) * I01 +
                   dy  *      dx  * I11)
    values[~valid] = 0.0
    return values

print("_bilinear_sample defined.")

In [ ]:
def warp_image(img, H):
    """
    Warp `img` using homography `H` via an inverse warp.

    The output bounding box is determined by projecting all four source corners
    through H.  Each RGB channel is warped independently.

    Parameters
    ----------
    img : (H, W) or (H, W, 3) uint8 / float array
    H   : (3, 3) homography  — maps source → destination coords

    Returns
    -------
    warped   : float64 image in the destination frame (values in [0, 1])
    offset   : (2,) int array [x_offset, y_offset] — top-left of the output
               in destination coordinate space
    """
    img = img.astype(np.float64)
    if img.max() > 1.0:
        img = img / 255.0
    h_src, w_src = img.shape[:2]
    is_color = img.ndim == 3

    # ── find output bounding box ──────────────────────────────
    corners_src = np.array(
        [[0,     0,     1],
         [w_src, 0,     1],
         [0,     h_src, 1],
         [w_src, h_src, 1]], dtype=float
    ).T                                         # (3, 4)
    corners_dst = H @ corners_src
    corners_dst /= corners_dst[2]

    x_min_i = int(np.floor(corners_dst[0].min()))
    x_max_i = int(np.ceil (corners_dst[0].max()))
    y_min_i = int(np.floor(corners_dst[1].min()))
    y_max_i = int(np.ceil (corners_dst[1].max()))

    out_w = x_max_i - x_min_i
    out_h = y_max_i - y_min_i
    offset = np.array([x_min_i, y_min_i])

    # ── build destination pixel grid ──────────────────────────
    xs = np.arange(out_w) + x_min_i
    ys = np.arange(out_h) + y_min_i
    grid_x, grid_y = np.meshgrid(xs, ys)          # (out_h, out_w)
    ones = np.ones_like(grid_x)
    pts_dst = np.stack([grid_x, grid_y, ones], axis=0).reshape(3, -1)  # (3, N)

    # ── inverse map: destination → source ────────────────────
    H_inv = np.linalg.inv(H)
    pts_src_mapped = H_inv @ pts_dst
    pts_src_mapped /= pts_src_mapped[2]
    src_x = pts_src_mapped[0]
    src_y = pts_src_mapped[1]

    # ── bilinear sample per channel ───────────────────────────
    if is_color:
        warped = np.zeros((out_h, out_w, img.shape[2]), dtype=np.float64)
        for c in range(img.shape[2]):
            vals = _bilinear_sample(img[:, :, c], src_x, src_y)
            warped[:, :, c] = vals.reshape(out_h, out_w)
    else:
        vals = _bilinear_sample(img, src_x, src_y)
        warped = vals.reshape(out_h, out_w)

    return warped, offset


warped_src, warp_offset = warp_image(img_src, H)
print(f"Warped shape : {warped_src.shape}")
print(f"Offset (x, y): {warp_offset}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(img_src);    axes[0].set_title("Source (original)");    axes[0].axis("off")
axes[1].imshow(warped_src); axes[1].set_title("Source (warped into dst frame)"); axes[1].axis("off")
plt.tight_layout()
plt.show()

---
## 2.3 — Create the Output Mosaic

In [ ]:
def create_mosaic(img_src, img_dst, H):
    """
    Warp img_src into img_dst's coordinate frame and overlay the two views.
    The destination image takes priority where both views overlap.
    Unoccupied regions are black.

    Parameters
    ----------
    img_src : source image (to be warped)
    img_dst : destination / reference image (stays fixed)
    H       : homography mapping img_src → img_dst coordinate frame

    Returns
    -------
    mosaic : float64 image, values in [0, 1]
    """
    img_src = img_src.astype(np.float64)
    img_dst = img_dst.astype(np.float64)
    if img_src.max() > 1.0: img_src /= 255.0
    if img_dst.max() > 1.0: img_dst /= 255.0

    warped, offset = warp_image(img_src, H)
    ox, oy = int(offset[0]), int(offset[1])

    h_dst, w_dst   = img_dst.shape[:2]
    h_warp, w_warp = warped.shape[:2]

    # canvas extent (destination coordinate system)
    x_min = min(0, ox);           y_min = min(0, oy)
    x_max = max(w_dst, ox+w_warp); y_max = max(h_dst, oy+h_warp)
    cw = x_max - x_min;           ch = y_max - y_min

    is_color = img_src.ndim == 3
    canvas = (np.zeros((ch, cw, img_src.shape[2]), dtype=np.float64)
              if is_color else np.zeros((ch, cw), dtype=np.float64))

    # paste warped source
    wy0, wx0 = oy - y_min, ox - x_min
    canvas[wy0:wy0+h_warp, wx0:wx0+w_warp] = warped

    # overlay destination (destination wins in overlap)
    dy0, dx0 = -y_min, -x_min
    dst_region = canvas[dy0:dy0+h_dst, dx0:dx0+w_dst]
    if is_color:
        mask = np.any(img_dst > 0, axis=-1)
    else:
        mask = img_dst > 0
    dst_region[mask] = img_dst[mask]
    canvas[dy0:dy0+h_dst, dx0:dx0+w_dst] = dst_region

    return np.clip(canvas, 0, 1)

print("create_mosaic defined.")

In [ ]:
mosaic = create_mosaic(img_src, img_dst, H)
print(f"Mosaic shape: {mosaic.shape}")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(img_src);  axes[0].set_title("Source");      axes[0].axis("off")
axes[1].imshow(img_dst);  axes[1].set_title("Destination"); axes[1].axis("off")
axes[2].imshow(mosaic);   axes[2].set_title("Mosaic");      axes[2].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
plt.imsave(OUTPUT_PATH, mosaic)
print(f"Mosaic saved to: {OUTPUT_PATH}")